# M5 Accident Severity Prediction Interface

This notebook loads the trained model selected by validation Macro-F1 and exposes a small inference interface for a future front end.

**Selected model:** FT-Transformer raw  
**Validation Macro-F1:** 0.4788  
**Test Macro-F1:** 0.4807  
**Test accuracy:** 0.7250

Main functions:

- `predict_accident_severity(record)` for one Python dictionary.
- `predict_accident_severity(records)` for a list of dictionaries or a pandas DataFrame.
- `get_input_schema()` for front-end field definitions and dropdown values.

The interface accepts the original 10 categorical fields and 4 base numeric fields. Missing-value handling, rare-category handling, time feature engineering, scaling, and tensor encoding are performed internally.

## 1. Imports and saved artifacts

Run this notebook in the same Python environment used for training. Required packages are `torch`, `pandas`, `numpy`, `scikit-learn`, and `joblib`.

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        required = candidate / "M5" / "model_outputs" / "ft_transformer_model.pt"
        if required.exists():
            return candidate
    raise FileNotFoundError(
        "Cannot find M5/model_outputs/ft_transformer_model.pt from the current directory."
    )


ROOT = find_project_root()
DATA_DIR = ROOT / "data" / "processed" / "M5"
MODEL_DIR = ROOT / "M5" / "model_outputs"

with (DATA_DIR / "preprocessing_metadata.json").open(encoding="utf-8") as file:
    META = json.load(file)
with (MODEL_DIR / "training_summary.json").open(encoding="utf-8") as file:
    TRAINING_SUMMARY = json.load(file)
with (MODEL_DIR / "ft_category_maps.json").open(encoding="utf-8") as file:
    CATEGORY_MAPS = json.load(file)

NUMERIC_SCALER = joblib.load(MODEL_DIR / "ft_numeric_scaler.joblib")

SELECTED_MODEL = TRAINING_SUMMARY["selected_model"]
if not SELECTED_MODEL.startswith("FT-Transformer"):
    raise RuntimeError(f"Saved winner is {SELECTED_MODEL!r}, not FT-Transformer.")

CAT_COLS = META["categorical_features"]
NUM_COLS = META["numeric_features"]
BASE_NUM_COLS = ["num_units", "crash_hour", "crash_day_of_week", "crash_month"]
INPUT_FEATURES = CAT_COLS + BASE_NUM_COLS
NUMERIC_MEDIANS = META["numeric_medians_from_train"]
RARE_CATEGORIES = {
    column: set(values)
    for column, values in META["rare_categories_from_train"].items()
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Selected model:", SELECTED_MODEL)
print("Inference device:", DEVICE)
print("Model directory:", MODEL_DIR)

## 2. Recreate the FT-Transformer architecture and load its weights

The architecture must match the training notebook exactly. This cell does not retrain or refit anything.

In [ ]:
class NumericalTokenizer(nn.Module):
    def __init__(self, n_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_features, d_token))
        self.bias = nn.Parameter(torch.empty(n_features, d_token))
        nn.init.xavier_uniform_(self.weight)
        nn.init.zeros_(self.bias)

    def forward(self, values):
        return values.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)


class FTTransformer(nn.Module):
    def __init__(
        self,
        category_sizes,
        n_numeric,
        n_classes,
        d_token=64,
        n_heads=8,
        n_layers=3,
        dropout=0.15,
    ):
        super().__init__()
        self.category_embeddings = nn.ModuleList([
            nn.Embedding(size, d_token) for size in category_sizes
        ])
        self.numeric_tokenizer = NumericalTokenizer(n_numeric, d_token)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        nn.init.normal_(self.cls_token, std=0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, d_token * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token * 2, n_classes),
        )

    def forward(self, categorical, numeric):
        categorical_tokens = torch.stack([
            embedding(categorical[:, index])
            for index, embedding in enumerate(self.category_embeddings)
        ], dim=1)
        numeric_tokens = self.numeric_tokenizer(numeric)
        cls = self.cls_token.expand(categorical.size(0), -1, -1)
        tokens = torch.cat([cls, categorical_tokens, numeric_tokens], dim=1)
        encoded = self.transformer(tokens)
        return self.head(encoded[:, 0])


MODEL_PATH = MODEL_DIR / "ft_transformer_model.pt"
try:
    CHECKPOINT = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
except TypeError:
    # Compatibility with PyTorch versions that do not have weights_only.
    CHECKPOINT = torch.load(MODEL_PATH, map_location=DEVICE)

TARGET_MAPPING = CHECKPOINT["target_mapping"]
ID_TO_LABEL = {int(class_id): label for label, class_id in TARGET_MAPPING.items()}
N_CLASSES = len(TARGET_MAPPING)

MODEL = FTTransformer(
    category_sizes=CHECKPOINT["category_sizes"],
    n_numeric=len(CHECKPOINT["numeric_features"]),
    n_classes=N_CLASSES,
    d_token=CHECKPOINT["d_token"],
    n_heads=CHECKPOINT["n_heads"],
    n_layers=CHECKPOINT["n_layers"],
    dropout=CHECKPOINT["dropout"],
).to(DEVICE)
MODEL.load_state_dict(CHECKPOINT["model_state_dict"])
MODEL.eval()

USE_CALIBRATION = SELECTED_MODEL.endswith("calibrated")
PROBABILITY_MULTIPLIERS = np.asarray(
    TRAINING_SUMMARY["selected_tuning"]["FT-Transformer"]["probability_multipliers"],
    dtype=np.float64,
)

print(f"Loaded {MODEL_PATH.name} successfully.")

## 3. Input preprocessing

This reproduces the training-time normalization, missing-value rules, rare-category fallback, numeric imputation, and cyclic time features. Unknown categorical values are safely routed to `OTHER_RARE` (or embedding index 0 if no fallback token exists).

In [ ]:
TEXT_PLACEHOLDERS = {
    "", "NAN", "NULL", "NONE", "UNKNOWN", "UNKNOWN/NA", "NOT APPLICABLE"
}


def _normalize_text(value):
    if pd.isna(value):
        return "MISSING"
    normalized = " ".join(str(value).strip().upper().split())
    return "MISSING" if normalized in TEXT_PLACEHOLDERS else normalized


def _normalize_category(column, value):
    normalized = _normalize_text(value)
    known_values = CATEGORY_MAPS[column]
    if normalized in RARE_CATEGORIES[column] or normalized not in known_values:
        normalized = "OTHER_RARE"
    return normalized


def _as_input_frame(records):
    if isinstance(records, dict):
        frame = pd.DataFrame([records])
        single_record = True
    elif isinstance(records, pd.DataFrame):
        frame = records.copy()
        single_record = False
    elif isinstance(records, (list, tuple)) and all(isinstance(row, dict) for row in records):
        frame = pd.DataFrame(records)
        single_record = False
    else:
        raise TypeError("Input must be a dict, list of dicts, tuple of dicts, or pandas DataFrame.")

    if frame.empty:
        raise ValueError("At least one input record is required.")
    missing_columns = [column for column in INPUT_FEATURES if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"Missing required input fields: {missing_columns}")
    return frame[INPUT_FEATURES].copy(), single_record


def _prepare_model_inputs(records):
    frame, single_record = _as_input_frame(records)

    for column in CAT_COLS:
        frame[column] = frame[column].map(lambda value: _normalize_category(column, value))

    for column in BASE_NUM_COLS:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    frame["num_units_was_missing"] = frame["num_units"].isna().astype("int8")
    for column in BASE_NUM_COLS:
        frame[column] = frame[column].fillna(NUMERIC_MEDIANS[column])

    if (frame["num_units"] < 0).any():
        raise ValueError("num_units must be greater than or equal to 0.")
    if not frame["crash_hour"].between(0, 23).all():
        raise ValueError("crash_hour must be between 0 and 23.")
    if not frame["crash_day_of_week"].between(1, 7).all():
        raise ValueError("crash_day_of_week must be between 1 and 7.")
    if not frame["crash_month"].between(1, 12).all():
        raise ValueError("crash_month must be between 1 and 12.")

    frame["is_weekend"] = frame["crash_day_of_week"].isin([1, 7]).astype("int8")
    frame["hour_sin"] = np.sin(2 * np.pi * frame["crash_hour"] / 24)
    frame["hour_cos"] = np.cos(2 * np.pi * frame["crash_hour"] / 24)
    frame["day_of_week_sin"] = np.sin(
        2 * np.pi * (frame["crash_day_of_week"] - 1) / 7
    )
    frame["day_of_week_cos"] = np.cos(
        2 * np.pi * (frame["crash_day_of_week"] - 1) / 7
    )
    frame["month_sin"] = np.sin(2 * np.pi * (frame["crash_month"] - 1) / 12)
    frame["month_cos"] = np.cos(2 * np.pi * (frame["crash_month"] - 1) / 12)

    categorical = np.column_stack([
        frame[column]
        .map(CATEGORY_MAPS[column])
        .fillna(0)
        .to_numpy(dtype=np.int64)
        for column in CAT_COLS
    ])
    numeric = NUMERIC_SCALER.transform(frame[NUM_COLS]).astype(np.float32)
    return categorical, numeric, single_record


## 4. Prediction and front-end schema interfaces

The return value contains only standard Python strings, integers, floats, dictionaries, and lists, so a later Flask/FastAPI/Streamlit layer can serialize it directly as JSON.

In [ ]:
def predict_accident_severity(records):
    """Predict one record or a batch of records.

    Parameters
    ----------
    records : dict, list[dict], tuple[dict], or pandas.DataFrame
        Each record must contain all fields listed in INPUT_FEATURES.

    Returns
    -------
    dict or list[dict]
        JSON-serializable prediction result(s).
    """
    categorical, numeric, single_record = _prepare_model_inputs(records)
    categorical_tensor = torch.as_tensor(categorical, dtype=torch.long, device=DEVICE)
    numeric_tensor = torch.as_tensor(numeric, dtype=torch.float32, device=DEVICE)

    with torch.inference_mode():
        logits = MODEL(categorical_tensor, numeric_tensor)
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()

    decision_scores = probabilities.copy()
    if USE_CALIBRATION:
        decision_scores *= PROBABILITY_MULTIPLIERS
    predicted_ids = decision_scores.argmax(axis=1)

    results = []
    for predicted_id, probability_row in zip(predicted_ids, probabilities):
        predicted_id = int(predicted_id)
        results.append({
            "predicted_class_id": predicted_id,
            "predicted_label": ID_TO_LABEL[predicted_id],
            "confidence": float(probability_row[predicted_id]),
            "probabilities": {
                ID_TO_LABEL[class_id]: float(probability_row[class_id])
                for class_id in sorted(ID_TO_LABEL)
            },
            "model": SELECTED_MODEL,
        })
    return results[0] if single_record else results


def get_input_schema():
    """Return JSON-serializable field metadata for a simple front end."""
    fields = []
    for column in CAT_COLS:
        options = sorted(
            value for value in CATEGORY_MAPS[column]
            if value not in {"MISSING", "OTHER_RARE"}
        )
        fields.append({
            "name": column,
            "type": "categorical",
            "required": True,
            "nullable": True,
            "options": options,
        })

    numeric_definitions = {
        "num_units": {"minimum": 0, "maximum": None},
        "crash_hour": {"minimum": 0, "maximum": 23},
        "crash_day_of_week": {"minimum": 1, "maximum": 7},
        "crash_month": {"minimum": 1, "maximum": 12},
    }
    for column in BASE_NUM_COLS:
        fields.append({
            "name": column,
            "type": "number",
            "required": True,
            "nullable": True,
            "default_if_null": float(NUMERIC_MEDIANS[column]),
            **numeric_definitions[column],
        })

    return {
        "model": SELECTED_MODEL,
        "input_fields": fields,
        "output_labels": [ID_TO_LABEL[class_id] for class_id in sorted(ID_TO_LABEL)],
    }


# Front-end developers can inspect this object to build form controls.
input_schema = get_input_schema()
print(f"Interface ready: {len(input_schema['input_fields'])} input fields")

## 5. Example: one prediction

The numeric conventions follow the source dataset: `crash_hour` is 0–23, `crash_day_of_week` is 1–7, and `crash_month` is 1–12.

In [ ]:
example_record = {
    "traffic_control_device": "TRAFFIC SIGNAL",
    "weather_condition": "CLEAR",
    "lighting_condition": "DAYLIGHT",
    "first_crash_type": "TURNING",
    "trafficway_type": "NOT DIVIDED",
    "alignment": "STRAIGHT AND LEVEL",
    "roadway_surface_cond": "DRY",
    "road_defect": "NO DEFECTS",
    "intersection_related_i": "Y",
    "prim_contributory_cause": None,
    "num_units": 3,
    "crash_hour": 12,
    "crash_day_of_week": 4,
    "crash_month": 10,
}

prediction = predict_accident_severity(example_record)
print(json.dumps(prediction, ensure_ascii=False, indent=2))

## 6. Optional smoke test against saved test data

This confirms that batch input works and that all probabilities are valid. It reads data only; it does not retrain the model.

In [ ]:
test_sample = pd.read_csv(DATA_DIR / "test_tabular.csv", nrows=5)
smoke_results = predict_accident_severity(test_sample[INPUT_FEATURES])

assert len(smoke_results) == len(test_sample)
for result in smoke_results:
    assert result["predicted_label"] in TARGET_MAPPING
    assert np.isclose(sum(result["probabilities"].values()), 1.0, atol=1e-6)

pd.DataFrame(smoke_results)

## Front-end integration note

A later web layer only needs to import or copy the model/interface cells once at process startup, then call `predict_accident_severity(request_json)` for each request. Do not reload the checkpoint for every prediction. For FastAPI, the returned dictionary can be returned directly from a POST endpoint; for Streamlit, pass the form values as `example_record` is passed above.